**Notebook 09: H3 DoS Analysis**


H3 Denial of Service Analysis

H3: Cybersecurity analytics reduces the operational impact of simulated denial-of-service conditions in RFID baggage tracking environments.

RQ3: How effective are cybersecurity analytics in reducing denial-of-service risks in baggage tracking?

This notebook evaluates H3 by comparing the same simulated denial of service activity under an unmitigated baseline condition and a store and forward mitigation condition using the prepared RFID baggage tracking data.

BLUF: The analysis examines whether cybersecurity analytics and the simulated store and forward continuity control reduce the operational impact of denial of service activity on the RFID baggage tracking record.

Step 1: Connect and Review the Prepared RFID Data

The prepared RFID baggage tracking file was connected from the Praxis project folder. The data was reviewed to confirm the number of scan records, baggage journeys, denial of service records, and available fields before the H3 comparison continued.

In [7]:
# Step 1: Connect and review the prepared RFID data


# Load the library

import pandas as pd


# Set the main project folder

import os

main_folder = os.getcwd()

# Check the project folder

if not os.path.isdir(main_folder + "/06_Prepared_Data"):
    main_folder = os.path.dirname(main_folder)


# Set the prepared data folder

prepared_folder = main_folder + "/06_Prepared_Data"


# Set the prepared RFID file

prepared_file = (
    prepared_folder +
    "/02_Prepared_RFID_Data.csv"
)


# Load the prepared RFID data

h3_data = pd.read_csv(prepared_file)


print("The prepared RFID data is connected.")


# Check the dataset size

print(
    "\nTotal RFID records:",
    len(h3_data)
)

print(
    "Total columns:",
    len(h3_data.columns)
)

print(
    "Total baggage journeys:",
    h3_data["bag_id"].nunique()
)

print(
    "Unique scan IDs:",
    h3_data["scan_id"].nunique()
)


# Check the denial of service records

print(
    "\nDenial of Service scans:",
    (h3_data["scan_status"] == "Denial of Service").sum()
)

print(
    "DoS disruption records:",
    (h3_data["disruption_window"] == 1).sum()
)

print(
    "Unique disruption IDs:",
    h3_data.loc[
        h3_data["disruption_window"] == 1,
        "disruption_id"
    ].nunique()
)


# Check the prepared fields

print("\nPrepared RFID fields:")

print(
    h3_data.columns.tolist()
)

The prepared RFID data is connected.

Total RFID records: 31575
Total columns: 34
Total baggage journeys: 5000
Unique scan IDs: 31575

Denial of Service scans: 75
DoS disruption records: 75
Unique disruption IDs: 10

Prepared RFID fields:
['bag_id', 'tag_id', 'reader_id', 'reader_location', 'custody_point', 'timestamp', 'expected_sequence', 'actual_sequence', 'expected_next_point', 'actual_next_point', 'scan_status', 'route_type', 'transfer_status', 'journey_status', 'journey_abnormal_type', 'expected_reader_id', 'expected_reader_location', 'expected_timestamp', 'disruption_id', 'disruption_window', 'flight_id', 'route_id', 'origin', 'destination', 'scan_id', 'disruption_minutes', 'reader_match', 'location_match', 'sequence_match', 'time_difference_minutes', 'delayed_indicator', 'scan_gap_minutes', 'final_label', 'spoofing_label']


Step 2: Establish the Unmitigated DoS Baseline

The unmitigated baseline represents the effect of the simulated denial of service condition when affected scan events are unavailable to the backend baggage tracking record.

The prepared data retains the original scan records, so a separate baseline working copy was created for H3. The 75 known denial of service events were treated as unavailable in the baseline condition while the original prepared RFID data remained unchanged.

The known disruption fields are used only to identify the controlled simulated DoS population. They are not used as evidence that the disruption was independently detected.

In [8]:
# Step 2: Establish the unmitigated DoS baseline


# Create a separate working copy for the baseline condition

h3_baseline = h3_data.copy()


# Identify the known simulated DoS events
#
# The disruption field identifies the controlled simulation
# population. It is not used as independent detection evidence.

dos_mask = h3_baseline["disruption_window"] == 1


# Label the H3 scenario condition

h3_baseline["scenario_condition"] = "Baseline"


# Assume scan events are available to the backend unless
# they are part of the simulated DoS disruption

h3_baseline["backend_available"] = True


# Mark the known DoS events as unavailable to the backend

h3_baseline.loc[
    dos_mask,
    "backend_available"
] = False


# Create the permanent event loss indicator

h3_baseline["permanent_loss_indicator"] = 0


h3_baseline.loc[
    dos_mask,
    "permanent_loss_indicator"
] = 1


# Create the backend view
#
# This represents the scan records that would remain available
# to the baggage tracking system during the baseline condition.

baseline_backend = h3_baseline[
    h3_baseline["backend_available"] == True
].copy()


# Create a journey level baseline result
#
# A baggage journey is considered incomplete when at least one
# required event is unavailable because of the simulated DoS condition.

baseline_journey_results = (
    h3_baseline
    .groupby("bag_id")
    .agg(
        baseline_permanent_losses=(
            "permanent_loss_indicator",
            "sum"
        )
    )
    .reset_index()
)


baseline_journey_results[
    "incomplete_custody_path"
] = (
    baseline_journey_results[
        "baseline_permanent_losses"
    ] > 0
)


# Review the baseline results

print("Baseline condition created.")


print(
    "\nTotal source RFID records:",
    len(h3_baseline)
)


print(
    "Backend available records:",
    len(baseline_backend)
)


print(
    "Backend unavailable DoS records:",
    (~h3_baseline["backend_available"]).sum()
)


print(
    "Baseline permanent event losses:",
    h3_baseline["permanent_loss_indicator"].sum()
)


print(
    "Baggage journeys with an incomplete custody path:",
    baseline_journey_results[
        "incomplete_custody_path"
    ].sum()
)


print(
    "Total baggage journeys:",
    baseline_journey_results["bag_id"].nunique()
)


print(
    "\nOriginal prepared data remains:",
    len(h3_data),
    "records"
)

Baseline condition created.

Total source RFID records: 31575
Backend available records: 31500
Backend unavailable DoS records: 75
Baseline permanent event losses: 75
Baggage journeys with an incomplete custody path: 75
Total baggage journeys: 5000

Original prepared data remains: 31575 records


Step 3: Apply the Store and Forward Mitigation

The mitigation condition uses the same 75 denial of service events evaluated in the baseline condition. Store and forward is applied to determine whether affected scan events can be temporarily retained and restored to the backend baggage tracking record after communications return.

The simulation uses a 90 percent buffering rate and a 95 percent restoration rate for buffered events. The original scan ID and scan timestamp remain unchanged. A separate recovery timestamp is used for restored events so the original baggage scan record is preserved.

The prepared data contains the duration of each simulated disruption but does not contain a separate communication restoration timestamp. For the H3 working condition, the stored disruption duration is added to the affected event timestamp to establish the recovery reference. This change is made only within the H3 mitigation dataset and does not change the prepared RFID file.

In [9]:
# Step 3: Apply the store and forward mitigation


# Load NumPy for the controlled simulation

import numpy as np


# Use the same random state identified in the methodology

np.random.seed(42)


# Create a separate working copy for the mitigation condition

h3_mitigation = h3_data.copy()


# Convert the scan timestamp to datetime

h3_mitigation["timestamp"] = pd.to_datetime(
    h3_mitigation["timestamp"]
)


# Identify the same known DoS events used in the baseline

dos_mask = h3_mitigation["disruption_window"] == 1

dos_index = h3_mitigation.index[dos_mask]


# Label the scenario condition

h3_mitigation["scenario_condition"] = "Mitigation"


# Preserve the original scan timestamp

h3_mitigation["original_scan_timestamp"] = (
    h3_mitigation["timestamp"]
)


# Create the mitigation outcome fields

h3_mitigation["buffer_indicator"] = 0

h3_mitigation["retransmitted_indicator"] = 0

h3_mitigation["recovery_indicator"] = 0

h3_mitigation["permanent_loss_indicator"] = 0

h3_mitigation["backend_available"] = True


# ---------------------------------------------------------
# Apply the 90 percent buffering condition
# ---------------------------------------------------------

buffer_draw = np.random.random(
    len(dos_index)
)


buffered_index = dos_index[
    buffer_draw < 0.90
]


h3_mitigation.loc[
    buffered_index,
    "buffer_indicator"
] = 1


# ---------------------------------------------------------
# Apply the 95 percent restoration condition
# to the events that entered the buffer
# ---------------------------------------------------------

restore_draw = np.random.random(
    len(buffered_index)
)


recovered_index = buffered_index[
    restore_draw < 0.95
]


h3_mitigation.loc[
    recovered_index,
    "retransmitted_indicator"
] = 1


h3_mitigation.loc[
    recovered_index,
    "recovery_indicator"
] = 1


# ---------------------------------------------------------
# Mark all affected DoS events as unavailable first
# ---------------------------------------------------------

h3_mitigation.loc[
    dos_index,
    "backend_available"
] = False


h3_mitigation.loc[
    dos_index,
    "permanent_loss_indicator"
] = 1


# ---------------------------------------------------------
# Restore the successfully recovered events
# ---------------------------------------------------------

h3_mitigation.loc[
    recovered_index,
    "backend_available"
] = True


h3_mitigation.loc[
    recovered_index,
    "permanent_loss_indicator"
] = 0


# ---------------------------------------------------------
# Establish when communication becomes available again
#
# The prepared file provides disruption duration but does
# not contain a separate restoration timestamp.
# ---------------------------------------------------------

h3_mitigation[
    "communication_restored_timestamp"
] = pd.NaT


h3_mitigation.loc[
    dos_index,
    "communication_restored_timestamp"
] = (
    h3_mitigation.loc[
        dos_index,
        "original_scan_timestamp"
    ]
    +
    pd.to_timedelta(
        h3_mitigation.loc[
            dos_index,
            "disruption_minutes"
        ],
        unit="m"
    )
)


# ---------------------------------------------------------
# Apply the 1 to 5 minute recovery period
# to successfully restored events
# ---------------------------------------------------------

h3_mitigation[
    "recovery_delay_minutes"
] = np.nan


recovery_delays = np.random.randint(
    1,
    6,
    size=len(recovered_index)
)


h3_mitigation.loc[
    recovered_index,
    "recovery_delay_minutes"
] = recovery_delays


# Create the recovery timestamp

h3_mitigation[
    "recovery_timestamp"
] = pd.NaT


h3_mitigation.loc[
    recovered_index,
    "recovery_timestamp"
] = (
    h3_mitigation.loc[
        recovered_index,
        "communication_restored_timestamp"
    ]
    +
    pd.to_timedelta(
        h3_mitigation.loc[
            recovered_index,
            "recovery_delay_minutes"
        ],
        unit="m"
    )
)


# ---------------------------------------------------------
# Calculate record staleness for restored DoS events
#
# Permanently lost events remain without a recovery
# timestamp and are not assigned an artificial staleness.
# ---------------------------------------------------------

h3_mitigation[
    "record_staleness_minutes"
] = np.nan


h3_mitigation.loc[
    recovered_index,
    "record_staleness_minutes"
] = (
    (
        h3_mitigation.loc[
            recovered_index,
            "recovery_timestamp"
        ]
        -
        h3_mitigation.loc[
            recovered_index,
            "original_scan_timestamp"
        ]
    )
    .dt.total_seconds()
    / 60
)


# ---------------------------------------------------------
# Review the mitigation results
# ---------------------------------------------------------

buffered_count = h3_mitigation.loc[
    dos_index,
    "buffer_indicator"
].sum()


recovered_count = h3_mitigation.loc[
    dos_index,
    "recovery_indicator"
].sum()


permanent_loss_count = h3_mitigation.loc[
    dos_index,
    "permanent_loss_indicator"
].sum()


print("Store and forward mitigation condition created.")


print(
    "\nTotal known DoS events:",
    len(dos_index)
)


print(
    "Events placed in the buffer:",
    buffered_count
)


print(
    "Events not placed in the buffer:",
    len(dos_index) - buffered_count
)


print(
    "Buffered events restored:",
    recovered_count
)


print(
    "Buffered events not restored:",
    buffered_count - recovered_count
)


print(
    "Permanent event losses after mitigation:",
    permanent_loss_count
)


print(
    "DoS events available to the backend after recovery:",
    h3_mitigation.loc[
        dos_index,
        "backend_available"
    ].sum()
)


print(
    "DoS events still unavailable to the backend:",
    (
        ~h3_mitigation.loc[
            dos_index,
            "backend_available"
        ]
    ).sum()
)


print(
    "\nMinimum recovery delay:",
    h3_mitigation.loc[
        recovered_index,
        "recovery_delay_minutes"
    ].min(),
    "minutes"
)


print(
    "Maximum recovery delay:",
    h3_mitigation.loc[
        recovered_index,
        "recovery_delay_minutes"
    ].max(),
    "minutes"
)


print(
    "\nOriginal prepared data remains:",
    len(h3_data),
    "records"
)

Store and forward mitigation condition created.

Total known DoS events: 75
Events placed in the buffer: 66
Events not placed in the buffer: 9
Buffered events restored: 64
Buffered events not restored: 2
Permanent event losses after mitigation: 11
DoS events available to the backend after recovery: 64
DoS events still unavailable to the backend: 11

Minimum recovery delay: 1.0 minutes
Maximum recovery delay: 5.0 minutes

Original prepared data remains: 31575 records


Step 4: Compare Baseline and Mitigation Performance

The baseline and store and forward results were compared using the H3 operational measures identified in the methodology. The comparison focuses on permanent event loss and incomplete custody paths, along with buffering, event restoration, recovery time, and record staleness for successfully recovered events.

A finite record staleness value is not assigned to permanently lost baseline events because those records never became available to the backend. This avoids creating an artificial recovery time for events that were not restored.

In [10]:
# Step 4: Compare baseline and mitigation performance


# Identify the same DoS population in both conditions

baseline_dos = h3_baseline[
    h3_baseline["disruption_window"] == 1
].copy()


mitigation_dos = h3_mitigation[
    h3_mitigation["disruption_window"] == 1
].copy()


# Confirm the same affected event population is used

same_event_population = (
    set(baseline_dos["scan_id"])
    ==
    set(mitigation_dos["scan_id"])
)


print(
    "Same DoS event population used:",
    same_event_population
)


# ---------------------------------------------------------
# Permanent event loss
# ---------------------------------------------------------

total_dos_events = len(baseline_dos)


baseline_permanent_loss = (
    baseline_dos[
        "permanent_loss_indicator"
    ].sum()
)


mitigation_permanent_loss = (
    mitigation_dos[
        "permanent_loss_indicator"
    ].sum()
)


baseline_loss_rate = (
    baseline_permanent_loss
    / total_dos_events
) * 100


mitigation_loss_rate = (
    mitigation_permanent_loss
    / total_dos_events
) * 100


permanent_loss_reduction = (
    (
        baseline_loss_rate
        -
        mitigation_loss_rate
    )
    /
    baseline_loss_rate
) * 100


# ---------------------------------------------------------
# Create the mitigation journey results
# ---------------------------------------------------------

mitigation_journey_results = (
    h3_mitigation
    .groupby("bag_id")
    .agg(
        mitigation_permanent_losses=(
            "permanent_loss_indicator",
            "sum"
        )
    )
    .reset_index()
)


mitigation_journey_results[
    "incomplete_custody_path"
] = (
    mitigation_journey_results[
        "mitigation_permanent_losses"
    ] > 0
)


# Identify the 75 DoS affected baggage journeys

dos_bag_ids = baseline_dos[
    "bag_id"
].unique()


baseline_affected_journeys = (
    baseline_journey_results[
        baseline_journey_results[
            "bag_id"
        ].isin(dos_bag_ids)
    ]
    .copy()
)


mitigation_affected_journeys = (
    mitigation_journey_results[
        mitigation_journey_results[
            "bag_id"
        ].isin(dos_bag_ids)
    ]
    .copy()
)


# ---------------------------------------------------------
# Incomplete custody paths
# ---------------------------------------------------------

total_dos_journeys = len(
    dos_bag_ids
)


baseline_incomplete_paths = (
    baseline_affected_journeys[
        "incomplete_custody_path"
    ].sum()
)


mitigation_incomplete_paths = (
    mitigation_affected_journeys[
        "incomplete_custody_path"
    ].sum()
)


baseline_incomplete_rate = (
    baseline_incomplete_paths
    / total_dos_journeys
) * 100


mitigation_incomplete_rate = (
    mitigation_incomplete_paths
    / total_dos_journeys
) * 100


incomplete_path_reduction = (
    (
        baseline_incomplete_rate
        -
        mitigation_incomplete_rate
    )
    /
    baseline_incomplete_rate
) * 100


# ---------------------------------------------------------
# Buffering and restoration performance
# ---------------------------------------------------------

buffered_events = (
    mitigation_dos[
        "buffer_indicator"
    ].sum()
)


restored_events = (
    mitigation_dos[
        "recovery_indicator"
    ].sum()
)


buffer_rate = (
    buffered_events
    / total_dos_events
) * 100


event_restoration_rate = (
    restored_events
    / buffered_events
) * 100


overall_event_recovery_rate = (
    restored_events
    / total_dos_events
) * 100


# ---------------------------------------------------------
# Recovery time
#
# Recovery time measures the time required to restore the
# buffered event after communication becomes available.
# ---------------------------------------------------------

recovered_events = mitigation_dos[
    mitigation_dos["recovery_indicator"] == 1
].copy()


mean_recovery_time = (
    recovered_events[
        "recovery_delay_minutes"
    ].mean()
)


# ---------------------------------------------------------
# Record staleness
#
# This value is calculated only for events that were
# successfully restored to the backend.
# ---------------------------------------------------------

mean_record_staleness = (
    recovered_events[
        "record_staleness_minutes"
    ].mean()
)


minimum_record_staleness = (
    recovered_events[
        "record_staleness_minutes"
    ].min()
)


maximum_record_staleness = (
    recovered_events[
        "record_staleness_minutes"
    ].max()
)


# ---------------------------------------------------------
# Create the baseline versus mitigation comparison
# ---------------------------------------------------------

h3_comparison = pd.DataFrame({

    "Measure": [
        "Permanent Event Loss",
        "Permanent Event Loss Rate",
        "Incomplete Custody Paths",
        "Incomplete Custody Path Rate"
    ],

    "Baseline": [
        baseline_permanent_loss,
        baseline_loss_rate,
        baseline_incomplete_paths,
        baseline_incomplete_rate
    ],

    "Mitigation": [
        mitigation_permanent_loss,
        mitigation_loss_rate,
        mitigation_incomplete_paths,
        mitigation_incomplete_rate
    ],

    "Reduction Percent": [
        permanent_loss_reduction,
        permanent_loss_reduction,
        incomplete_path_reduction,
        incomplete_path_reduction
    ]
})


h3_comparison[
    [
        "Baseline",
        "Mitigation",
        "Reduction Percent"
    ]
] = h3_comparison[
    [
        "Baseline",
        "Mitigation",
        "Reduction Percent"
    ]
].round(2)


# ---------------------------------------------------------
# Create the mitigation recovery summary
# ---------------------------------------------------------

h3_recovery_summary = pd.DataFrame({

    "Measure": [
        "Total DoS Events",
        "Buffered Events",
        "Buffer Rate",
        "Restored Events",
        "Event Restoration Rate",
        "Overall Event Recovery Rate",
        "Permanent Event Losses",
        "Mean Recovery Time",
        "Mean Record Staleness"
    ],

    "Result": [
        total_dos_events,
        buffered_events,
        buffer_rate,
        restored_events,
        event_restoration_rate,
        overall_event_recovery_rate,
        mitigation_permanent_loss,
        mean_recovery_time,
        mean_record_staleness
    ]
})


h3_recovery_summary[
    "Result"
] = h3_recovery_summary[
    "Result"
].round(2)


# ---------------------------------------------------------
# Display the H3 results
# ---------------------------------------------------------

print(
    "\nBaseline versus mitigation comparison:"
)

display(h3_comparison)


print(
    "\nStore and forward recovery summary:"
)

display(h3_recovery_summary)


print(
    "\nMinimum record staleness:",
    round(
        minimum_record_staleness,
        2
    ),
    "minutes"
)


print(
    "Maximum record staleness:",
    round(
        maximum_record_staleness,
        2
    ),
    "minutes"
)


print(
    "\nBaseline record staleness:"
)


print(
    "Not assigned because the affected baseline events remained permanently unavailable to the backend."
)

Same DoS event population used: True

Baseline versus mitigation comparison:


,Measure,Baseline,Mitigation,Reduction Percent
0,Permanent Event Loss,75.0,11.00,85.33
1,Permanent Event Loss Rate,100.0,14.67,85.33
2,Incomplete Custody Paths,75.0,11.00,85.33
3,Incomplete Custody Path Rate,100.0,14.67,85.33



Store and forward recovery summary:


,Measure,Result
0,Total DoS Events,75.00
1,Buffered Events,66.00
2,Buffer Rate,88.00
3,Restored Events,64.00
4,Event Restoration Rate,96.97
5,Overall Event Recovery Rate,85.33
6,Permanent Event Losses,11.00
7,Mean Recovery Time,2.70
8,Mean Record Staleness,23.34



Minimum record staleness: 18.0 minutes
Maximum record staleness: 34.0 minutes

Baseline record staleness:
Not assigned because the affected baseline events remained permanently unavailable to the backend.


Step 5: Validate the H3 Results and Determine the H3 Decision

The final H3 review confirms that the same denial of service events were evaluated under both conditions and that the original prepared RFID data remained unchanged. The baseline and mitigation results were then reviewed using permanent event loss, incomplete custody paths, event restoration, recovery time, and record staleness.

A numerical reduction in baseline record staleness is not calculated because the affected baseline events remained permanently unavailable to the backend. For the mitigation condition, record staleness is reported only for events that were successfully restored.

The H3 decision is based on whether the store and forward condition reduced the operational effect of the simulated denial of service activity while restoring affected baggage scan records within the controlled simulation.

In [11]:
# Step 5: Validate the H3 results and determine the H3 decision


# ---------------------------------------------------------
# Validate the event population
# ---------------------------------------------------------

same_event_population = (
    set(baseline_dos["scan_id"])
    ==
    set(mitigation_dos["scan_id"])
)


same_event_count = (
    len(baseline_dos)
    ==
    len(mitigation_dos)
    ==
    75
)


source_data_unchanged = (
    len(h3_data) == 31575
    and
    h3_data["scan_id"].nunique() == 31575
)


# ---------------------------------------------------------
# Validate the mitigation accounting
# ---------------------------------------------------------

mitigation_accounting_check = (
    restored_events
    +
    mitigation_permanent_loss
    ==
    total_dos_events
)


buffer_accounting_check = (
    buffered_events
    +
    (
        total_dos_events
        -
        buffered_events
    )
    ==
    total_dos_events
)


# ---------------------------------------------------------
# Validate the recovery period
# ---------------------------------------------------------

recovery_delay_check = (
    recovered_events[
        "recovery_delay_minutes"
    ].min() >= 1
    and
    recovered_events[
        "recovery_delay_minutes"
    ].max() <= 5
)


# ---------------------------------------------------------
# Validate the direction of the H3 results
# ---------------------------------------------------------

permanent_loss_improved = (
    mitigation_permanent_loss
    <
    baseline_permanent_loss
)


custody_path_improved = (
    mitigation_incomplete_paths
    <
    baseline_incomplete_paths
)


events_were_restored = (
    restored_events > 0
)


finite_recovery_observed = (
    recovered_events[
        "recovery_delay_minutes"
    ].notna().all()
)


# ---------------------------------------------------------
# Create the validation table
# ---------------------------------------------------------

h3_validation = pd.DataFrame({

    "Validation Check": [
        "Same DoS event population",
        "Same DoS event count",
        "Original prepared data unchanged",
        "Mitigation event accounting",
        "Buffer event accounting",
        "Recovery delay within 1 to 5 minutes",
        "Permanent event loss improved",
        "Incomplete custody paths improved",
        "Affected events were restored",
        "Recovered events have finite recovery time"
    ],

    "Result": [
        same_event_population,
        same_event_count,
        source_data_unchanged,
        mitigation_accounting_check,
        buffer_accounting_check,
        recovery_delay_check,
        permanent_loss_improved,
        custody_path_improved,
        events_were_restored,
        finite_recovery_observed
    ]
})


print("H3 validation results:")

display(h3_validation)


# ---------------------------------------------------------
# Determine the H3 decision
#
# The decision uses the direction of the operational
# measures defined for H3. No separate DoS detection
# accuracy claim is made from the known scenario labels.
# ---------------------------------------------------------

if (
    same_event_population
    and
    source_data_unchanged
    and
    permanent_loss_improved
    and
    custody_path_improved
    and
    events_were_restored
    and
    finite_recovery_observed
):

    h3_decision = "Supported"

elif (
    permanent_loss_improved
    or
    custody_path_improved
    or
    events_were_restored
):

    h3_decision = "Partially Supported"

else:

    h3_decision = "Not Supported"


# ---------------------------------------------------------
# Display the final H3 measures
# ---------------------------------------------------------

print("\nFinal H3 Results")


print(
    "\nBaseline permanent event losses:",
    int(baseline_permanent_loss)
)


print(
    "Mitigation permanent event losses:",
    int(mitigation_permanent_loss)
)


print(
    "Permanent event loss reduction:",
    round(
        permanent_loss_reduction,
        2
    ),
    "%"
)


print(
    "\nBaseline incomplete custody paths:",
    int(baseline_incomplete_paths)
)


print(
    "Mitigation incomplete custody paths:",
    int(mitigation_incomplete_paths)
)


print(
    "Incomplete custody path reduction:",
    round(
        incomplete_path_reduction,
        2
    ),
    "%"
)


print(
    "\nBuffered events:",
    int(buffered_events),
    "of",
    total_dos_events
)


print(
    "Buffer rate:",
    round(
        buffer_rate,
        2
    ),
    "%"
)


print(
    "\nRestored buffered events:",
    int(restored_events),
    "of",
    int(buffered_events)
)


print(
    "Event restoration rate:",
    round(
        event_restoration_rate,
        2
    ),
    "%"
)


print(
    "Overall event recovery rate:",
    round(
        overall_event_recovery_rate,
        2
    ),
    "%"
)


print(
    "\nMean recovery time:",
    round(
        mean_recovery_time,
        2
    ),
    "minutes"
)


print(
    "Mean record staleness for restored events:",
    round(
        mean_record_staleness,
        2
    ),
    "minutes"
)


print(
    "Record staleness range:",
    round(
        minimum_record_staleness,
        2
    ),
    "to",
    round(
        maximum_record_staleness,
        2
    ),
    "minutes"
)


print(
    "\nBaseline record staleness:"
)


print(
    "Not assigned because baseline DoS events remained permanently unavailable."
)


print(
    "\nIndependent DoS detection claim:"
)


print(
    "Not made from the known scenario labels. The H3 comparison evaluates operational impact under the controlled DoS condition."
)


print(
    "\nH3 Decision:",
    h3_decision
)

H3 validation results:


,Validation Check,Result
0,Same DoS event population,True
1,Same DoS event count,True
2,Original prepared data unchanged,True
3,Mitigation event accounting,True
4,Buffer event accounting,True
5,Recovery delay within 1 to 5 minutes,True
6,Permanent event loss improved,True
7,Incomplete custody paths improved,True
8,Affected events were restored,True
9,Recovered events have finite recovery time,True



Final H3 Results

Baseline permanent event losses: 75
Mitigation permanent event losses: 11
Permanent event loss reduction: 85.33 %

Baseline incomplete custody paths: 75
Mitigation incomplete custody paths: 11
Incomplete custody path reduction: 85.33 %

Buffered events: 66 of 75
Buffer rate: 88.0 %

Restored buffered events: 64 of 66
Event restoration rate: 96.97 %
Overall event recovery rate: 85.33 %

Mean recovery time: 2.7 minutes
Mean record staleness for restored events: 23.34 minutes
Record staleness range: 18.0 to 34.0 minutes

Baseline record staleness:
Not assigned because baseline DoS events remained permanently unavailable.

Independent DoS detection claim:
Not made from the known scenario labels. The H3 comparison evaluates operational impact under the controlled DoS condition.

H3 Decision: Supported


In [12]:

# Save the H3 results to GitHub Codespaces

# Create the H3 results folder
results_folder = main_folder + "/09_H3_Results"

os.makedirs(results_folder, exist_ok=True)

# Save the baseline and mitigation comparison
h3_comparison.to_csv(
    results_folder + "/h3_comparison.csv",
    index=False
)

# Save the recovery results
h3_recovery_summary.to_csv(
    results_folder + "/h3_recovery_summary.csv",
    index=False
)

# Save the validation results
h3_validation.to_csv(
    results_folder + "/h3_validation.csv",
    index=False
)

print("H3 results saved successfully.")


H3 results saved successfully.


H3 Summary

The H3 analysis compared the same 75 simulated denial of service events under an unmitigated baseline condition and a store and forward mitigation condition.

Under the baseline condition, all 75 affected events remained unavailable to the backend baggage tracking record, resulting in 75 permanent event losses and 75 incomplete custody paths. After store and forward was applied, permanent event losses decreased to 11 and incomplete custody paths decreased to 11. Both measures were reduced by 85.33 percent.

Of the 75 affected events, 66 were placed in the temporary buffer and 64 were successfully restored. This resulted in a 96.97 percent restoration rate for buffered events and an overall recovery rate of 85.33 percent. The mean recovery time after communication returned was 2.70 minutes.

For successfully restored events, mean record staleness was 23.34 minutes, with values ranging from 18 to 34 minutes. Record staleness was not assigned to permanently lost baseline events because those records never became available to the backend.

The known denial of service fields were used to identify the controlled simulated disruption population and were not treated as evidence of independent disruption detection.

Based on the reduction in permanent event loss and incomplete custody paths, along with the high event restoration rate and measurable recovery performance, the results support H3.